In [51]:
import random
from pokerkit import *
from dotenv import load_dotenv
import json
from anthropic import Anthropic, beta_tool
from mediumterm import MediumTermMemory
from longterm import LongTermMemory
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
import re
import time

In [52]:
load_dotenv()

client = Anthropic()

## Functions

### Agentic tool use functions

In [53]:
PLAYER_LABEL_MAP = dict(zip([i for i in range(1,9)], [f"P{i+1}" for i in range(1,9)])) #our opponents, for example seat 1 has P2, logged as P2.txt


def build_system_prompt(ltm: LongTermMemory, active_player_ids: list[str]) -> str:
    """Construct the system prompt, injecting LTM profiles for known opponents."""

    base = (
        "You are a poker agent playing 3-handed Texas Hold'em No-Limit.\n"
        "Blinds are 1000/2000. You are Player 1 (Hero). Player 2 is P2, Player 3 is P3 and so on.\n"
        "Think about pot odds, position, hand strength, and how many chips you have remaining before deciding. "
        "Make sure to carefully read and use the information in the provided json.\n"
        "If you are unsure of your hand strength, use the get_equity_strength tool to calculate your win probability. If you use this tool, you must then call the take_action tool.\n"
        "Remember that the equity tool only tells you strength of the cards, not the behavior of your opponents.\n"
        "Use self-notes to reflect on your past actions, and opponent notes to reflect on the behavior of your opponents.\n"
        "Use the take_action tool to submit your decision. You must ALWAYS end your turn by using this tool."
    )

    self_profile = ltm.read_self_profile()
    ltm_section = (
        "\n\n--- YOUR STRATEGIC SELF-NOTES (from long-term memory) ---\n"
        + self_profile
        + "\n--- END SELF NOTES ---")

    opponent_sections = []
    for pid in active_player_ids:
        profile = ltm.lookup_player(pid)
        opponent_sections.append(f"\n--- LONG-TERM PROFILE: {pid} ---\n{profile}\n--- END PROFILE: {pid} ---")
    if opponent_sections:
        ltm_section += "\n" + "\n".join(opponent_sections)
    return base + ltm_section


def extract_tags_from_reasoning(reasoning_texts: list[str]) -> dict[str, list[str]]:
    tag_map: dict[str, list[str]] = {}

    for text in reasoning_texts:
        if not text:
            continue
        words = text.split()
        for i, word in enumerate(words):
            player_label = word.strip(",:").upper()
            if not (player_label.startswith("P") and player_label[1:].isdigit()):
                continue
            if player_label[1] not in "123456789": #! we only support opponent names P1 through P9
                continue
            #search for rating
            lookahead = words[i+1 : i+11]
            for chunk in lookahead:
                if "/10" not in chunk:
                    continue
                score_part = chunk.split("/10")[0].strip()
                if not score_part.isdigit():
                    break
                score = int(score_part)
                if score >= 7:
                    tag = "aggressive"
                elif score <= 3:
                    tag = "passive"
                else:
                    tag = "balanced"
                tag_map.setdefault(player_label, []).append(tag)
                break
    return {pid: [Counter(tags).most_common(1)[0][0]] for pid, tags in tag_map.items()}

def run_post_hand_reflection(ltm: LongTermMemory, eval_text: str, active_player_ids: list[str], reasonings: list, payoffs: tuple) -> None:
    """
    Called once after each hand to:
    1. Write per-opponent observations to LTM.
    2. Append a self-note about how Hero played.
    """
    if not eval_text: return

    reasoning_texts = [r[2] for r in reasonings if len(r) >= 3]
    tag_map = extract_tags_from_reasoning(reasoning_texts)
    for pid in active_player_ids:
        tags = tag_map.get(pid, [])
        ltm.log_player_update(player_id=pid, observation=eval_text, tags=tags)
    hero_payoff = payoffs[0] if payoffs else 0
    outcome = "won" if hero_payoff > 0 else ("lost" if hero_payoff < 0 else "broke even")
    hero_reasoning = " | ".join(f"{reasoning[0]} {reasoning[1] if reasoning[1] else ''}: {reasoning[2]}"
        for reasoning in reasonings if len(reasoning) >= 3)
    self_note = (
        f"Hand outcome: {outcome} ({hero_payoff:+,} chips). "
        f"Opponent eval: {eval_text[:300]}. "
        f"Hero reasoning summary: {hero_reasoning[:400]}."
    )
    ltm.log_self_update(observation=self_note, section="Lessons learned")


@beta_tool
def take_action(action: str, reasoning: str, amount: float = 0) -> str:
    """Submit your poker action for this turn and give a maximum 3 sentence explanation for why.

    Args:
        action: One of fold, check, call, raise
        amount: Raise amount in chips. Required if action is raise.
        reasoning: Reasoning for why you took a poker action. Maximum 3 sentences. Rate how aggressive each opponent's actions are on a 1-10 scale, but only if you made it to the flop or later. 
    """
    return json.dumps({"status": "accepted", "action": action, "amount": amount, 'reasoning': reasoning})

@beta_tool
def get_equity_strength(hole_cards: str, board_cards: str, active_players: int) -> str:
    """Calculates the mathematical win probability (equity) of your current hand using Monte Carlo simulations.
    Call this tool when you are facing a difficult decision, a large bet, or are unsure of your exact hand strength.
    
    Args:
        hole_cards: Your two hole cards as a string (e.g., 'AsKc').
        board_cards: The visible board cards as a string (e.g., 'Td9h2c'). Leave empty '' for preflop.
        active_players: Total number of players dealt into the simulation.
    """
    
    print(f"\n   [TOOL TRIGGERED] Calculating equity for {hole_cards} | Board: {board_cards} | Players: {active_players}")
    
    board_input = tuple(Card.parse(board_cards)) if board_cards else ()
    
    calculated_equity = calculate_hand_strength(
        active_players,      
        parse_range(hole_cards),
        board_input,
        active_players,      
        5,                
        Deck.STANDARD,
        (StandardHighHand,),
        sample_count=500
    )
    
    win_percentage = round(calculated_equity * 100, 1)
    
    return json.dumps({
        "status": "success", 
        "equity": calculated_equity,
        "message": f"You have a {win_percentage}% chance of winning."
    })


def run_turn(state_json, ltm: LongTermMemory, active_player_ids: list[str]):
    system_prompt = build_system_prompt(ltm, active_player_ids)

    messages = [{
        "role": "user",
        "content": f"""It's your turn to act. Here is the current game state:
    
    ```json
    {state_json}
    ```
    
    Decide your action."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[take_action, get_equity_strength],
        system=system_prompt,
        messages=messages
    )

    agent_action = None
    tokens = ""

    for message in runner:
        tokens = f"Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}"
        for block in message.content:
            if block.type == "tool_use" and block.name == "take_action":
                agent_action = block.input

    return agent_action, tokens

def street_converter(index):
    if index == 0:
        return 'preflop'
    elif index == 1:
        return 'flop'
    elif index == 2:
        return 'turn'
    elif index == 3:
        return 'river'
    
def get_visible_operations(state):
    visible = []
    showdown = sum(1 for p in state.statuses if p) > 1
    winners = set()
    for op in state.operations:
        if isinstance(op, ChipsPushing):
            winners = {i for i, amt in enumerate(op.amounts) if amt > 0}

    for op in state.operations:
        if isinstance(op, HoleDealing):
            continue
        elif isinstance(op, CardBurning):
            continue
        elif isinstance(op, HoleCardsShowingOrMucking):
            if showdown or op.player_index in winners:
                visible.append(op)
        else:
            visible.append(op)
    return visible

judge_system_prompt = """You are an evaluating agent for players playing Texas Hold-em. Your job is to evaluate and provide useful information about the other non-agent players to aid Player 1 in defeating them. Do not evaluate Player 1 (Hero), focus on the other players, denoted P2 and up. Remember that index 0 is the agent, index 1 is Player 2 and so on. Use the generate_observation tool to submit your decision."""

@beta_tool
def generate_observation(evaluation: str) -> str:
    """Evaluate the action history of the game in the previous poker hand. 

    Args:
        evaluation: The evaluation of how the other players played. Be specific as to any mistakes or good moves each non-Hero player made. 
    """
    return json.dumps({"status": "accepted", "evaluation": evaluation})

    
def run_observation_generator(action_history):
    eval_json = {'action_history': action_history}
    messages = [{
        "role": "user",
        "content": f"""Here is the action history of the game. 
    
    ```json
    {eval_json}
    ```
    
    Generate your observations."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[generate_observation],
        system=judge_system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        for block in message.content:
            if block.type == "tool_use" and block.name == "generate_observation":
                agent_action = block.input["evaluation"]

    return agent_action


def _position_name(seat):
    if seat == 0:
        return "SB"
    if seat == 1:
        return "BB"
    return "other"


def humanize_action_history(ops, active_indices, players):
    history = []
    for op in ops:
        s = str(op)
        def repl(m):
            seat = int(m.group(1))
            return f'player={players[active_indices[seat]]}'
        s = re.sub(r'player_index=(\d+)', repl, s)
        history.append(s)
    return history


### Bots

In [54]:
def ev_based_strategy(state, player_index, ev_pot_threshold=0.5, num_simulations=500):
    """
    Bets/raises when equity > ev_pot_threshold, calls when pot odds justify it,
    else folds.
    """
    hole = state.hole_cards[player_index]

    if len(hole) < 2:
        if state.can_check_or_call():
            state.check_or_call()
        elif state.can_fold():
            state.fold()
        return

    active_players = sum(1 for s in state.statuses if s)
    board_input = tuple(c for street in state.board_cards for c in street)
    hole_range = {frozenset(hole)}

    equity = calculate_hand_strength(
        active_players,
        hole_range,
        board_input,
        2,                    # hole cards per player (Holdem)
        5,                    # total board cards (Holdem)
        Deck.STANDARD,
        (StandardHighHand,),
        sample_count=num_simulations,
    )

    pot = state.total_pot_amount or 1

    if equity > ev_pot_threshold and state.can_complete_bet_or_raise_to():
        lo = state.min_completion_betting_or_raising_to_amount
        hi = state.max_completion_betting_or_raising_to_amount
        scale = min(equity, 1.0)
        target = int(lo + scale * (hi - lo))
        target = max(lo, min(target, hi))
        state.complete_bet_or_raise_to(target)
    elif state.can_check_or_call():
        call_amount = state.checking_or_calling_amount or 0
        if call_amount == 0 or equity > call_amount / (pot + call_amount):
            state.check_or_call()
        elif state.can_fold():
            state.fold()
        else:
            state.check_or_call()
    elif state.can_fold():
        state.fold()
    else:
        state.check_or_call()



def pocket_pair_strategy(state, player_index):
    hole = state.hole_cards[player_index]
    is_pair = len(hole) >= 2 and hole[0].rank == hole[1].rank
    strong_pair_ranks = {"A", "K", "Q", "J", "T", "9", "8", "7"}
    is_strong_pair = is_pair and hole[0].rank in strong_pair_ranks
    if is_strong_pair and state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif is_pair and state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()
    else:
        state.check_or_call()

def punter_strategy(state, player_index):
    punter_is_punting_this_hand = random.random() < 0.1
    if not punter_is_punting_this_hand:
        if state.can_fold():
            state.fold()
        else:
            state.check_or_call()
        return
    if state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

def tight_passive_strategy(state, player_index):
    r = random.random()
    if r < 0.9 and state.can_fold():
        state.fold()
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

def random_raiser(state, player_index):
    action_rand = random.random()
    if action_rand < 0.1 and state.can_complete_bet_or_raise_to():
    # 10% chance to raise to 3x the big blind
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount # All-in

    # Raise to 2x the minimum required to keep it moving
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif action_rand < 0.2 and state.can_fold():
    # 10% chance to fold
        state.fold()
    else:
        state.check_or_call()

# Edit this to change what roles each opponent plays
opponent_strategies = {
    1: pocket_pair_strategy,
    2: punter_strategy,
    3: tight_passive_strategy,
    4: random_raiser,
    5: ev_based_strategy,
    6: lambda st,id: ev_based_strategy(st, id, ev_pot_threshold=0.7)} #6 is more conservative.

### Result plotting

In [55]:
def plotter(stacks, game_number, players, colors):
    df = pd.DataFrame(stacks, columns=players)
    hand_number = len(df)
    win_pct = (df.Agent.diff() > 0).sum() / df.Agent.diff().notna().sum()
    hands = np.arange(1, hand_number+1, 1)
    plt.figure(figsize=[16, 9])
    if colors is None:
        colors = ['red', 'orange', 'green', 'blue', 'indigo']
    for player, color in zip(players, colors):
        plt.plot(hands, df[player], color=color, label=player)
    plt.xlim(1, hand_number)
    plt.xlabel('Hand #')
    plt.ylabel('Chip Count')
    handles, labels = plt.gca().get_legend_handles_labels()
    handles.append(Patch(facecolor="none", edgecolor="none"))
    labels.append(f"Win %: {round(win_pct*100, 2)}%")
    plt.legend(handles=handles, labels=labels, prop={'size': 15})
    plt.title(f'Game Number {game_number}')
    plt.savefig(f'results/run_{game_number}.png', bbox_inches='tight')
    plt.show()

### Game Loop

In [56]:
def run_game(game_id, players, player_stacks, sb=1000, bb=2000, min_bet=2000, opponent_strategies=opponent_strategies):
    med = MediumTermMemory()
    med.new_game(game_id=game_id, players=players)
    ltm = LongTermMemory()
    ltm.new_session(game_id=game_id)
    
    # Initial Setup
    hand_count = 0
    stacks_per_hand = []
    button = len(player_stacks) - 1
    # The Global Game Loop: Runs until only one player has chips
    while len([s for s in player_stacks if s > 0]) > 1:
        time.sleep(10)
        reasonings = []
        active_indices = [i for i, stack in enumerate(player_stacks) if stack > 0]
    
        if len(active_indices) < 2:
            break
        
        sb_global = next((i for i in active_indices if i > button), active_indices[0])
        pivot = active_indices.index(sb_global)
        active_indices = active_indices[pivot:] + active_indices[:pivot]
        
        # Map engine seat indices → human player IDs for LTM lookups
        # Seat 0 = Hero; seat 1+ = opponents in position order
        active_player_ids = [
            PLAYER_LABEL_MAP[i] for i in active_indices if i in PLAYER_LABEL_MAP
        ]
    
        current_stacks = tuple(player_stacks[i] for i in active_indices)
        hand_count += 1
        print(f"\n--- STARTING HAND {hand_count} ---")
        print(f'Button: {button}', active_indices)

        state = NoLimitTexasHoldem.create_state(
            (
                Automation.ANTE_POSTING,
                Automation.BET_COLLECTION,
                Automation.BLIND_OR_STRADDLE_POSTING,
                Automation.CARD_BURNING,
                Automation.HOLE_DEALING,
                Automation.BOARD_DEALING,
                Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
                Automation.HAND_KILLING,
                Automation.CHIPS_PUSHING,
                Automation.CHIPS_PULLING,
            ),
            True, 0, (sb, bb), min_bet, tuple(current_stacks), len(active_indices)
        )
    
        # Per-Turn Decision Loop
        while state.status:
            if state.can_deal_hole():
                state.deal_hole()
            elif state.can_deal_board():
                state.deal_board()
    
            elif state.actor_index is not None:
                global_player_index = active_indices[state.actor_index]
                res = get_visible_operations(state)
                obs = {
                    "your_index": state.actor_index,
                    "pot": state.total_pot_amount,
                    "position": _position_name(state.actor_index),
                    "board": state.board_cards,
                    "hole": state.hole_cards[state.actor_index],
                    "stacks": state.stacks,
                    "bets": state.bets,
                    "street": street_converter(state.street_index),
                    "can_fold?": state.can_fold(),
                    "can_check_or_call?": state.can_check_or_call(),
                    "can_raise?": state.can_complete_bet_or_raise_to(),
                    "min_raise": state.min_completion_betting_or_raising_to_amount,
                    "max_raise": state.max_completion_betting_or_raising_to_amount,
                    "how_much_to_call": state.checking_or_calling_amount,
                    "action_history": res
                }
    
                if global_player_index == 0: #hero/agent
                    action, tokens = run_turn(obs, ltm, active_player_ids)
                    
                    if action is None:
                        print("!! run_turn returned None")
                        print(f"   obs: {obs}")
                        print(f"   hand {hand_count}, street {state.street_index}")
                        raise RuntimeError("run_turn returned None — inspect logs")

                    if action['action'] in ['check', 'call']:
                        state.check_or_call()
                    elif action['action'] == 'raise':
                        raw = action.get('amount')
                        low = state.min_completion_betting_or_raising_to_amount
                        high = state.max_completion_betting_or_raising_to_amount
                        clamped = max(low, min(raw, high))
                        if clamped != raw:
                            print(f"!! clamped raise: {raw} -> {clamped} (range [{low}, {high}])")
                        state.complete_bet_or_raise_to(clamped)
                    elif action['action'] == 'fold':
                        state.fold()
                    reasonings.append([action['action'], action.get('amount', 0), action.get('reasoning', '')])
    
                else:  # Random bots
                    opponent_strategies[global_player_index](state, state.actor_index)
            else:
                break
    
        payoffs_by_player = [0] * len(player_stacks)
        for seat, global_idx in enumerate(active_indices):
            payoffs_by_player[global_idx] = int(state.payoffs[seat])
        ops = get_visible_operations(state)
        history = humanize_action_history(ops, active_indices, players)
        med.ingest_hand(action_history=history, reasoning=reasonings,
            chip_changes=payoffs_by_player)
        eval_text = run_observation_generator(history)
        med.log_trend(eval_text)
    
        run_post_hand_reflection(ltm=ltm, eval_text=eval_text, active_player_ids=active_player_ids, reasonings=reasonings, payoffs=payoffs_by_player)
        
        for g in range(len(player_stacks)):
            player_stacks[g] += payoffs_by_player[g]
        stacks_per_hand.append(player_stacks.copy())
        
        print(f"Hand {hand_count} Results: {payoffs_by_player}")
        print(f"New Stacks: {player_stacks}")
        
        button = (button + 1) % len(player_stacks)
        while player_stacks[button] <= 0:
            button = (button + 1) % len(player_stacks)
    ltm.close_session() #flush to disk
    
    print(f"\nGAME OVER after {hand_count} hands!")
    print(f"Final Winner Stacks: {player_stacks}")
    print(f"\nLTM index now contains: {list(ltm._index.keys())}")
    return stacks_per_hand

In [57]:
res = []
for i in range(2,5):
    res.append(run_game('00'+str(i), ['Hero', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7'], [20_000, 20_000, 20_000, 20_000, 20_000, 20_000, 20_000]))


--- STARTING HAND 1 ---
Button: 6 [0, 1, 2, 3, 4, 5, 6]

   [TOOL TRIGGERED] Calculating equity for QdKs | Board: 5hKd6h | Players: 2

   [TOOL TRIGGERED] Calculating equity for QdKs | Board: 5hKd6h4d | Players: 2
Hand 1 Results: [9500, -2000, 0, 0, -7500, 0, 0]
New Stacks: [29500, 18000, 20000, 20000, 12500, 20000, 20000]

--- STARTING HAND 2 ---
Button: 0 [1, 2, 3, 4, 5, 6, 0]

   [TOOL TRIGGERED] Calculating equity for QsQc | Board: 7h7dKh | Players: 3
Hand 2 Results: [34500, -1000, -2000, 0, -12500, 0, -19000]
New Stacks: [64000, 17000, 18000, 20000, 0, 20000, 1000]

--- STARTING HAND 3 ---
Button: 1 [2, 3, 5, 6, 0, 1]
Hand 3 Results: [0, 0, 2000, -2000, 0, 0, 0]
New Stacks: [64000, 17000, 20000, 18000, 0, 20000, 1000]

--- STARTING HAND 4 ---
Button: 2 [3, 5, 6, 0, 1, 2]

   [TOOL TRIGGERED] Calculating equity for TsTc | Board:  | Players: 2


Error occurred while calling tool: get_equity_strength
Traceback (most recent call last):
  File "C:\Documents\econ_project\.venv\Lib\site-packages\anthropic\lib\tools\_beta_runner.py", line 344, in _generate_tool_call_response
    result = tool.call(tool_use.input)
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Documents\econ_project\.venv\Lib\site-packages\anthropic\lib\tools\_beta_functions.py", line 241, in call
    return self._func_with_validate(**cast(Any, input))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Documents\econ_project\.venv\Lib\site-packages\pydantic\_internal\_validate_call.py", line 40, in wrapper_function
    return wrapper(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Documents\econ_project\.venv\Lib\site-packages\pydantic\_internal\_validate_call.py", line 137, in __call__
    res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


   [TOOL TRIGGERED] Calculating equity for TsIc | Board: 7c2d5c | Players: 2

   [TOOL TRIGGERED] Calculating equity for TsTc | Board: 7c2d5c | Players: 2

   [TOOL TRIGGERED] Calculating equity for TsTs | Board: 7c2d5cAc | Players: 2
Hand 4 Results: [-20000, 0, 0, -1000, 0, 21000, 0]
New Stacks: [44000, 17000, 20000, 17000, 0, 41000, 1000]

--- STARTING HAND 5 ---
Button: 3 [5, 6, 0, 1, 2, 3]
Hand 5 Results: [-1000, 3000, 0, 0, 0, -1000, -1000]
New Stacks: [43000, 20000, 20000, 17000, 0, 40000, 0]

--- STARTING HAND 6 ---
Button: 5 [0, 1, 2, 3, 5]

   [TOOL TRIGGERED] Calculating equity for QsQc | Board: JcTs5s | Players: 4

   [TOOL TRIGGERED] Calculating equity for Qs8c | Board: JcTs5s | Players: 4

   [TOOL TRIGGERED] Calculating equity for 8cQs | Board: JcTs5s3hTd | Players: 3
Hand 6 Results: [-2000, -2000, 0, 4000, 0, 0, 0]
New Stacks: [41000, 18000, 20000, 21000, 0, 40000, 0]

--- STARTING HAND 7 ---
Button: 0 [1, 2, 3, 5, 0]
Hand 7 Results: [0, -1000, 1000, 0, 0, 0, 0]
New Sta

RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 50,000 input tokens per minute (org: 37404817-637b-41df-b353-a1baf0bb1018, model: claude-haiku-4-5-20251001). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Reduce the prompt length or the maximum tokens requested, or try again later. View your current limits at https://console.anthropic.com/settings/limits. To raise this limit, purchase credits to advance to the next usage tier at https://console.anthropic.com/settings/billing."}, 'request_id': 'req_011CbYeVPXH9Tkzc5mVuVVEx'}